In [1]:
import pandas as pd
import sklearn.model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder

In [2]:
mushroom_data = pd.read_csv("mushrooms.csv")

In [3]:
mushroom_data.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [4]:
mushroom_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   class                     8124 non-null   object
 1   cap-shape                 8124 non-null   object
 2   cap-surface               8124 non-null   object
 3   cap-color                 8124 non-null   object
 4   bruises                   8124 non-null   object
 5   odor                      8124 non-null   object
 6   gill-attachment           8124 non-null   object
 7   gill-spacing              8124 non-null   object
 8   gill-size                 8124 non-null   object
 9   gill-color                8124 non-null   object
 10  stalk-shape               8124 non-null   object
 11  stalk-root                8124 non-null   object
 12  stalk-surface-above-ring  8124 non-null   object
 13  stalk-surface-below-ring  8124 non-null   object
 14  stalk-color-above-ring  

In [5]:
(mushroom_data["class"] == "e").sum()

4208

Out of 8124 total cases, we have 4208 (52%) that are edible and 3916 (48%) that are poisonous. This is a balanced dataset.

In [6]:
(mushroom_data["stalk-root"] == "?").sum()

2480

The stalk root feature "missing" must mean that the data is not available. It would simplify things if we could drop the entire feature.
Let's try dropping it and if that reduces the efficacy of our model we will try something else.

In [7]:
(mushroom_data == "?").sum()

class                          0
cap-shape                      0
cap-surface                    0
cap-color                      0
bruises                        0
odor                           0
gill-attachment                0
gill-spacing                   0
gill-size                      0
gill-color                     0
stalk-shape                    0
stalk-root                  2480
stalk-surface-above-ring       0
stalk-surface-below-ring       0
stalk-color-above-ring         0
stalk-color-below-ring         0
veil-type                      0
veil-color                     0
ring-number                    0
ring-type                      0
spore-print-color              0
population                     0
habitat                        0
dtype: int64

Split into dependent and independent variable.

In [8]:
mushroom_y = mushroom_data["class"]
mushroom_x = mushroom_data.drop(["class", "stalk-root"], axis=1)

mushroom_x = mushroom_x.to_numpy()
mushroom_y = mushroom_y.to_numpy().reshape(-1, 1)

All variables are nominal or ordinal. Use OneHotEncoder to make dummy numeric columns. I wanted to use OrdinalEncoder for some of the features but I couldn't figure out how to use them on only some of the columns in time.

In [9]:
ohe = OneHotEncoder(sparse=False)
dummy_y = ohe.fit_transform(mushroom_y)

In [10]:
print(dummy_y.shape, dummy_y, sep="\n")

(8124, 2)
[[0. 1.]
 [1. 0.]
 [1. 0.]
 ...
 [1. 0.]
 [0. 1.]
 [1. 0.]]


In [11]:
dummy_x = ohe.fit_transform(mushroom_x)
print(dummy_x.shape, dummy_x, sep="\n")

(8124, 112)
[[0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


Split into training and testing data with a 20:80 split.

In [12]:
x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(
    dummy_x,
    dummy_y,
    train_size=0.2,
)

In [13]:
print(y_test.shape, y_train.shape)

(6500, 2) (1624, 2)


In [14]:
logmodel = LogisticRegression()
logmodel.fit(x_train, y_train[:, 1])
predictions = logmodel.predict(x_test)

In [15]:
print(classification_report(y_test[:, 1], predictions))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      3382
         1.0       1.00      1.00      1.00      3118

    accuracy                           1.00      6500
   macro avg       1.00      1.00      1.00      6500
weighted avg       1.00      1.00      1.00      6500

